# Analiza jakości danych w systemie

## 1. Wprowadzenie

### 1.1 Cel analizy

Celem niniejszej analizy jest ocena jakości danych zgromadzonych w hurtowni oraz identyfikacja potencjalnych problemów mogących wpłynąć na wiarygodność prowadzonych analiz.

### 1.2 Zakres analizy

Analiza obejmuje dane z sezonów 2007/2008 - 2025/2026, z 50 lig piłkarskich z 25 krajów. Sezon 2025/2026 jest traktowany jako sezon bieżący i nie jest uwzględniany w analizach kompletności.

### 1.3 Metodologia

Przedstawione analizy opierają się na danych pobranych zarówno ze znormalizowanej części hurtowni jak i w niektórych przypadkach z systemu źródłowego. Do weryfikacji jakości wykorzystano zestaw metryk obejmujących kompletność, spójność oraz poprawność danych.

### 1.4 Konfiguracja

In [ ]:
from sqlalchemy import create_engine, text
import urllib
from sqlalchemy import inspect
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.set_palette("deep")

In [ ]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=127.0.0.1,1433;"
    "DATABASE=LEAGUE_DB;"
    "UID=sa;"
    "PWD=YourStrongPassword1!;"
    "TrustServerCertificate=yes;"
)

source_engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

source_inspector = inspect(source_engine)

In [ ]:
tables = source_inspector.get_table_names()
tables
t = pd.read_sql("""SELECT TOP(20) ht_goals_team_a, ht_goals_team_b FROM dbo.T_F_Match_Stats""", source_engine)
t

In [ ]:
destination_engine = create_engine(f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:" \
    f"{os.environ['POSTGRES_PASSWORD']}@localhost:{os.environ['POSTGRES_PORT']}" \
    f"/{os.environ['POSTGRES_DB']}"
)

In [ ]:
d = pd.read_sql("""SELECT COUNT(*) FROM core.teams WHERE common_name = 'Lille'""", destination_engine)
d

## 2. Kompletność danych historycznych

### 2.1 Ogólna charakterystyka zbioru danych

In [ ]:
sl_query = 'SELECT COUNT(*) FROM dbo.T_DIM_League;'
sl_df = pd.read_sql(sl_query, source_engine)
sl_count = sl_df.iloc[0, 0]

dl_query = 'SELECT COUNT(*) FROM core.leagues;'
dl_df = pd.read_sql(dl_query, destination_engine)
dl_count = dl_df.iloc[0, 0]

sm_query = 'SELECT COUNT(*) FROM dbo.T_F_Match_Stats;'
sm_df = pd.read_sql(sm_query, source_engine)
sm_count = sm_df.iloc[0, 0]

dm_query = 'SELECT COUNT(*) FROM core.matches;'
dm_df = pd.read_sql(dm_query, destination_engine)
dm_count = dm_df.iloc[0, 0]

st_query = 'SELECT COUNT(*) FROM dbo.T_DIM_Team;'
st_df = pd.read_sql(st_query, source_engine)
st_count = st_df.iloc[0, 0]

dt_query = 'SELECT COUNT(*) FROM core.teams;'
dt_df = pd.read_sql(dt_query, destination_engine)
dt_count = dt_df.iloc[0, 0]

sp_query = 'SELECT COUNT(*) FROM dbo.T_DIM_Player;'
sp_df = pd.read_sql(sp_query, source_engine)
sp_count = sp_df.iloc[0, 0]

dp_query = 'SELECT COUNT(*) FROM core.players;'
dp_df = pd.read_sql(dp_query, destination_engine)
dp_count = dp_df.iloc[0, 0]

sr_query = 'SELECT COUNT(*) FROM dbo.T_DIM_Referees;'
sr_df = pd.read_sql(sr_query, source_engine)
sr_count = sr_df.iloc[0, 0]

dr_query = 'SELECT COUNT(*) FROM core.referees;'
dr_df = pd.read_sql(dr_query, destination_engine)
dr_count = dr_df.iloc[0, 0]

plot_df = pd.DataFrame([
    { 
        "tabela": "Ligi",
        "źródło": sl_count,
        "edw": dl_count
    },
    { 
        "tabela": "Drużyny",
        "źródło": st_count,
        "edw": dt_count
    },
    { 
        "tabela": "Mecze",
        "źródło": sm_count,
        "edw": dm_count
    },
    { 
        "tabela": "Zawodnicy",
        "źródło": sp_count,
        "edw": dp_count
    },
    { 
        "tabela": "Sędziowie",
        "źródło": sr_count,
        "edw": dr_count
    }
])

plot_long = plot_df.melt(id_vars="tabela", value_vars=["źródło", "edw"], 
                         var_name="baza danych", value_name="liczba_rekordów")

plt.figure(figsize=(10,6))
ax = sns.barplot(data=plot_long, x="tabela", y="liczba_rekordów", hue="baza danych")
plt.yscale("log")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height):,}', 
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', fontsize=9, rotation=0)

plt.title("Porównanie liczby rekordów: źródło vs docelowa baza")
plt.ylabel("Liczba rekordów")
plt.xlabel("Tabela")
plt.show()


In [ ]:
source_date_range_query = """
SELECT 
    MIN(DateSK) AS source_min_date, 
    MAX(DateSK) AS source_max_date
FROM 
    dbo.T_F_Match_Stats;
"""
sdr_df = pd.read_sql(source_date_range_query, source_engine)

destination_date_range_query = """
SELECT 
    MIN(date) AS destination_min_date, 
    MAX(date) AS destination_max_date
FROM 
    core.matches;
"""
ddr_df = pd.read_sql(destination_date_range_query, destination_engine)

df = pd.concat([sdr_df, ddr_df], axis=1)
df


In [ ]:
source_unique_seasons = """
SELECT 
    COUNT(DISTINCT(SeasonSK)) AS source_season_count
FROM 
    dbo.T_F_Match_Stats;
""" 
sus_df = pd.read_sql(source_unique_seasons, source_engine)

destination_unique_seasons = """
SELECT 
    COUNT(*) AS destination_season_count
FROM 
    core.seasons;
""" 
dus_df = pd.read_sql(destination_unique_seasons, destination_engine)


df = pd.DataFrame({
    "source_season_count": [sus_df.iloc[0, 0]],
    "destination_season_count": [dus_df.iloc[0, 0]],
})
df

In [ ]:
source_unique_leagues = """
SELECT 
    COUNT(DISTINCT(LeagueName)) AS source_unique_leagues_count
FROM 
    dbo.T_DIM_League;
""" 
sul_df = pd.read_sql(source_unique_leagues, source_engine)

destination_unique_leagues = """
SELECT 
    COUNT(*) AS destination_unique_leagues_count
FROM 
    core.leagues;
""" 
dul_df = pd.read_sql(destination_unique_leagues, destination_engine)

df = pd.DataFrame({
    "source_unique_leagues_count": [sul_df.iloc[0, 0]],
    "destination_unique_leagues_count": [dul_df.iloc[0, 0]],
})
df

### 2.2 Pokrycie sezonów

In [ ]:
source_matches_per_season = """
    SELECT 
        s.SeasonName AS sezon,
        COUNT(ms.MatchSK) AS źródło
    FROM
        dbo.T_F_Match_Stats ms
    JOIN 
        dbo.T_DIM_Season s ON ms.SeasonSK = s.SeasonSK
    GROUP BY
        s.SeasonName;
"""
smps_df = pd.read_sql(source_matches_per_season, source_engine)

destination_matches_per_season = """
    SELECT 
        s.name AS sezon,
        COUNT(m.match_id) AS edw
    FROM
        core.matches m
    JOIN 
        core.seasons s ON s.season_id = m.season_id 
    GROUP BY
        s.name;
"""
dmps_df = pd.read_sql(destination_matches_per_season, destination_engine)

df = pd.merge(smps_df, dmps_df, on="sezon", how="outer")

plot_df = df.melt(id_vars="sezon", value_vars=["źródło", "edw"], 
                  var_name="baza danych", value_name="liczba meczy")

plt.figure(figsize=(12,6))
sns.barplot(data=plot_df, x="sezon", y="liczba meczy", hue="baza danych")
plt.xticks(rotation=90)
plt.title("Porównanie liczby meczów per sezon")
plt.tight_layout()
plt.show()


### 2.3 Pokrycie lig

In [ ]:
source_matches_per_league = """
    SELECT 
        ln.League_name AS liga,
        COUNT(m.MatchSK) AS źródło
    FROM
        dbo.T_F_Match_Stats m
    JOIN
        dbo.T_DIM_League l ON l.competition_id = m.competition_id
    JOIN
        dbo.T_DIM_League_name ln ON ln.competition_id = m.competition_id
    GROUP BY
        ln.League_name;
"""
smpl_df = pd.read_sql(source_matches_per_league, source_engine)

destination_matches_per_league = """
    SELECT
        l.key_name AS liga,
        COUNT(DISTINCT m.match_id) AS edw
    FROM core.matches m
    JOIN core.teams_leagues_seasons tls
        ON tls.team_id = m.home_team_id
        AND tls.season_id = m.season_id
    JOIN core.leagues l
    ON l.league_id = tls.league_id
    GROUP BY l.key_name;
"""
dmpl_df = pd.read_sql(destination_matches_per_league, destination_engine)

df = pd.merge(
    smpl_df,
    dmpl_df,
    on="liga",
    how="outer"
).fillna(0)

df[["źródło", "edw"]] = df[["źródło", "edw"]].astype(int)

plot_df = df.melt(
    id_vars="liga",
    value_vars=["źródło", "edw"],
    var_name="baza danych",
    value_name="liczba meczów"
)

league_order = (
    df.assign(max_cnt=df[["źródło", "edw"]].max(axis=1))
      .sort_values("max_cnt", ascending=False)["liga"]
      .tolist()
)

plt.figure(figsize=(14, 6))
sns.barplot(
    data=plot_df,
    x="liga",
    y="liczba meczów",
    hue="baza danych",
    order=league_order
)

plt.xticks(rotation=90, ha="right")
plt.ylabel("Liczba meczów")
plt.xlabel("Liga")
plt.title("Porównanie liczby meczów per liga")
plt.legend(title="Baza danych")
plt.tight_layout()
plt.show()

## 3. Analiza kompletności danych

In [ ]:
dm_query = """
    SELECT * FROM core.matches;
"""
dm_df = pd.read_sql(dm_query, destination_engine)
missing_percent = dm_df.isna().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)

missing_df = pd.DataFrame({'Kolumna': missing_percent.index, 'Procent braków': missing_percent.values})

plt.figure(figsize=(12,6))
sns.heatmap(dm_df.isna(), cbar=False, yticklabels=False, cmap='Blues')
plt.title('Mapa cieplna braków danych dla tabeli meczy')
plt.xlabel('Kolumny')
plt.show()

In [ ]:
dm_query = """
    SELECT * FROM core.teams;
"""
dm_df = pd.read_sql(dm_query, destination_engine)
missing_percent = dm_df.isna().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)

missing_df = pd.DataFrame({'Kolumna': missing_percent.index, 'Procent braków': missing_percent.values})

plt.figure(figsize=(12,6))
sns.heatmap(dm_df.isna(), cbar=False, yticklabels=False, cmap='Blues')
plt.title('Mapa cieplna braków danych dla tabeli drużyn')
plt.xlabel('Kolumny')
plt.show()

In [ ]:
dm_query = """
    SELECT * FROM core.players;
"""
dm_df = pd.read_sql(dm_query, destination_engine)
missing_percent = dm_df.isna().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)

missing_df = pd.DataFrame({'Kolumna': missing_percent.index, 'Procent braków': missing_percent.values})

plt.figure(figsize=(12,6))
sns.heatmap(dm_df.isna(), cbar=False, yticklabels=False, cmap='Blues')
plt.title('Mapa cieplna braków danych dla tabeli zawodników')
plt.xlabel('Kolumny')
plt.show()

In [ ]:
dm_query = """
    SELECT * FROM core.referees;
"""
dm_df = pd.read_sql(dm_query, destination_engine)
missing_percent = dm_df.isna().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)

missing_df = pd.DataFrame({'Kolumna': missing_percent.index, 'Procent braków': missing_percent.values})

plt.figure(figsize=(12,6))
sns.heatmap(dm_df.isna(), cbar=False, yticklabels=False, cmap='Blues')
plt.title('Mapa cieplna braków danych dla tabeli sędziów')
plt.xlabel('Kolumny')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tables = {
    "mecze": "core.matches",
    "zawodnicy": "core.players",
    "sedziowie": "core.referees",
    "druzyny": "core.teams"
}

plot_data = []

for table_name, table_sql in tables.items():
    df = pd.read_sql(f"SELECT * FROM {table_sql};", destination_engine)
    missing_percent = df.isna().mean() * 100
    missing_df = pd.DataFrame({
        "Kolumna": missing_percent.index,
        "Procent_braków": missing_percent.values,
        "Tabela": table_name
    })
    missing_df = missing_df[missing_df["Procent_braków"] > 0]
    top_missing = missing_df.sort_values("Procent_braków", ascending=False).head(5)
    plot_data.append(top_missing)

plot_df = pd.concat(plot_data, ignore_index=True)

plt.figure(figsize=(12,6))
ax = sns.barplot(
    data=plot_df, 
    x='Kolumna', 
    y='Procent_braków', 
    hue='Tabela',
)

plt.xticks(rotation=45, ha='right')
plt.title('Top kolumn z największym % braków w głównych tabelach')
plt.ylabel('% braków')
plt.xlabel('')

plt.legend(title='Tabela', loc='upper right')

# dodanie labeli nad słupkami
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(
            p.get_x() + p.get_width()/2.,  
            height + 1,                    
            f'{height:.1f}%',
            ha='center',
            va='bottom',
            fontsize=9
        )

plt.tight_layout()
plt.show()
